In [ ]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

<table align="left">
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/colab/import/https:%2F%2Fraw.githubusercontent.com%2Fgooglemaps-samples%2Finsights-samples%2Fmain%2Fcustom_satellite_embeddings%2Fnotebooks%2Fee_data_exploration%2FCustom_Satellite_Embeddings_Data_Exploration.ipynb?utm_source=custom_satellite_embeddings_notebooks">
      <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Google Cloud Colab Enterprise logo"><br> Open in Colab Enterprise
    </a>
  </td>
</table>

# Custom Satellite Embeddings Data Exploration

Custom satellite embeddings imagery is delivered as [Cloud Optimized Geotiffs](https://cogeo.org/) (COGs) in [Google Cloud Storage](https://cloud.google.com/storage).  COGs in Cloud Storage can be loaded into [Google Earth Engine](https://earthengine.google.com) for visualization and analysis.  They can be loaded directly using [ee.Image.loadGeoTIFF](https://developers.google.com/earth-engine/apidocs/ee-image-loadgeotiff).  Or you can create Earth Engine `Image` assets backed by COGs in Cloud Storage. The advantage of creating COG-backed assets is that indices on spatial and metadata properties make collections of COG-backed assets performant.  See [this guide](https://developers.google.com/earth-engine/Earth_Engine_asset_from_cloud_geotiff) for information on creating COG-backed Earth Engine assets.  This guide assumes that Earth Engine `ImageCollection`s of COG-backed images are already created.

In this guide, you can explore a time series of six monthly satellite embeddings images in an agricultural area of the California, USA central valley.  Like the [annual embeddings](https://developers.google.com/earth-engine/datasets/catalog/GOOGLE_SATELLITE_EMBEDDING_V1_ANNUAL), each image has 64 information-dense bands.

## Earth Engine Authentication and Initialization

To use Earth Engine, you must have a Cloud Project with the Earth Engine API activated.

In [ ]:
import ee
import geemap

from IPython.display import Image, display
from pprint import pprint

In [ ]:
# Your project with the Earth Engine API activated.
PROJECT = "YOUR-PROJECT"

In [ ]:
# Initialize Earth Engine (Assumes you have already authenticated via gcloud)
try:
  ee.Initialize(project=PROJECT)
except Exception:
  ee.Authenticate()
  ee.Initialize(project=PROJECT)

## Load the satellite embeddings `ImageCollection`

This sample is a time series of monthly images, centered on the 15th of the month, and spanning six months between 2025-11-15 and 2026-05-15.  The data covers an area of ~18,000 hectares of mostly cropland in California.

In [ ]:
meloland_california_2026_crop_cycles = ee.ImageCollection(
    "projects/mldp-trusted-testers-external/assets/alphaearth_foundations/preview/demos/meloland_california_2026_crop_cycles")

## Examine a single image

Use `getInfo()` to request information about the Earth Engine object to the client (this notebook).  This will print information about the properties (metadata) of the image, bands of the image and backing COGs.

In [ ]:
pprint(meloland_california_2026_crop_cycles.first().getInfo())

## Spatial mosaicking

Since Satellite embeddings are tiled, first aggregate the collection spatially, such that each image in the collection represents one time.  Here `mosaic()` is used to spatially aggregate the imagery.

In [ ]:
# Spatial mosaicking.
time_starts = meloland_california_2026_crop_cycles.aggregate_histogram('system:time_start')

def mosaic_function(time):
  time_start = ee.Number.parse(time)
  date = ee.Date(time_start)
  filtered = meloland_california_2026_crop_cycles.filterDate(date)
  # Transfer time and geographic properties.
  return filtered.mosaic().set('system:time_start', time_start).clip(filtered.geometry(100))

c = ee.ImageCollection.fromImages(time_starts.keys().map(mosaic_function))

## Make a multi-temporal RGB visualization of one band

Each satellite embeddings image has 64 bands and each band may or may not have a physical interpretation.  The following is just a visualization to illustrate change over the six month period. Use the `geemap` library for display.

In [ ]:
# Three 64D temporal slices.
r = c.filterDate('2025-11-01', '2025-12-31').mean()
g = c.filterDate('2026-01-01', '2026-02-28').mean()
b = c.filterDate('2026-03-01', '2026-04-30').mean()
# Band 45 RGB.
b_index = 45
rgb1 = ee.Image.cat(r.select(b_index), g.select(b_index), b.select(b_index))

Display the image interactively with `geemap`.

In [ ]:
Map = geemap.Map(center=(32.8287, -115.45131), zoom=13)
Map.addLayer(rgb1, {"min": -0.36, "max": 0.03}, "rgb1")
Map

Or request a thumbnail URL, which you can display using an IPython widget, download with `curl`, copy/paste, etc.  These URLs are temporary and should not be used for long term access.

In [ ]:
thumb_url = rgb1.clip(c.first().geometry()).getThumbUrl({"min": -0.36, "max": 0.03, "dimensions": "512x512"})
display(Image(url=thumb_url))

## Make a multi-temporal RGB visualization of change

Here we'll represent change as the dot product (similarity) between mean slices of the time series and a reference year.  To create a visualization, use the three dot products (from different times) as three bands in an RGB image.  Redder pixels indicate greater similarity to reference conditions at the start of the period, greener pixels indicate greater similarity to reference conditions in the middle of the period, etc.

Note that the 64D satellite embeddings are length-1 until you take the mean of a bunch of vectors.  The functions needed to compute those dot products are defined here.

In [ ]:
def dot(im1: ee.Image, im2: ee.Image) -> ee.Image:
  """Dot product of length-1 images."""
  return im1.multiply(im2).reduce("sum")


def scaled_dot(im1: ee.Image, im2: ee.Image) -> ee.Image:
  """Dot product of images that might not be length 1."""
  return dot(im1, im2).divide(dot(im1, im1).sqrt()).divide(dot(im2, im2).sqrt())


# Similarities of the means to a reference year (2024).
aef = ee.ImageCollection("GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL")
aef_2024 = aef.filter(ee.Filter.calendarRange(2024, 2024, 'year')).mean()
rgb2 = ee.Image.cat(
  scaled_dot(aef_2024, r),
  scaled_dot(aef_2024, g),
  scaled_dot(aef_2024, b)
)
Map.addLayer(rgb2, {"min": 0.42, "max": 0.95}, "rgb2")
Map

## Make a multi-temporal RGB visualization of similarity

To find the pixels most similar to a single location over time, first compute the reference embedding at a single location.  Then compute the dot product between that one reference and the pixels in the three temporal slices. Use a point in an arbitrary field in the image.

In [ ]:
point = ee.Geometry.Point([-115.47235642112162, 32.83620978393891])
Map.addLayer(point, {}, "point")

In [ ]:
# One (user specified) point overlaid on the first image.
mean_aef = c.first().reduceRegion(
  reducer="mean",
  geometry=point,
  scale=10
)

# Mean similarities to the one point.
similarities = c.map(
    lambda image: scaled_dot(image, mean_aef.toImage()).set('system:time_start', image.date().millis()))
rgb3 = ee.Image.cat(
  similarities.filterDate('2025-11-01', '2025-12-31').mean(),
  similarities.filterDate('2026-01-01', '2026-02-28').mean(),
  similarities.filterDate('2026-03-01', '2026-04-30').mean())
Map.addLayer(rgb3, {"min": 0.2, "max": 0.8}, "rgb3")
Map

## What's next?

Explore [other Custom Satellite Embeddings samples](https://developers.devsite.corp.google.com/maps/documentation/custom-satellite-embeddings/samples).